# Atelier Préparation de Données Tabulaires

## Contexte

Une entreprise exploite plusieurs bâtiments intelligents équipés de capteurs IoT. Chaque capteur collecte régulièrement des informations sur la température, l'humidité, la qualité de l'air, la consommation énergétique, le nombre de personnes présentes, le type de bâtiment, le mode de fonctionnement et l'état du système de climatisation.

Les données collectées sont destinées à alimenter ultérieurement un modèle de Machine Learning capable de prédire la consommation énergétique ou de détecter les situations anormales.

Cependant, les données brutes présentent volontairement différents problèmes : valeurs manquantes, doublons, valeurs aberrantes, types incorrects, valeurs incohérentes, variables catégorielles, catégories rares, déséquilibre des classes et échelles différentes entre variables.

L'objectif de l'atelier est donc de transformer le fichier brut en un jeu de données propre et prêt pour le Machine Learning.

# Partie 1 – Explorer les données

In [40]:
# Question 1 : Charger les données CSV
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/smart_building_raw.csv")

In [41]:
# Question 2 : Afficher les premières lignes du dataset
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


In [42]:
# Question 3 : Afficher les dernières lignes du dataset
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


In [43]:
# Question 4 : Nombre d'observations du dataset
obs = df.shape[0]
print(f"Nombre d'observations : {obs}")

Nombre d'observations : 507


In [44]:
# Question 5 : Nombre de variables du dataset
nbre_variables = df.shape[1]
print(f"Nombre de variables : {nbre_variables}")

Nombre de variables : 14


In [45]:
# Question 6 : Identifier les variables numériques
variable_numerique = df.select_dtypes(include=[np.number]).columns
print(f"Variables numériques : {variable_numerique}")

Variables numériques : Index(['id_mesure', 'temperature', 'humidite', 'co2', 'occupation',
       'consommation_kwh'],
      dtype='object')


In [46]:
# Question 7 : Identifier les variables catégorielles
variable_categorique = df.select_dtypes(include=['object']).columns
print(f"Variables catégorielles : {variable_categorique}")

Variables catégorielles : Index(['date', 'batiment', 'type_batiment', 'zone', 'mode_climatisation',
       'etat_systeme', 'jour_semaine', 'alerte'],
      dtype='object')


In [47]:
# Question 8 : Identifier les colonnes de type date
df.dtypes

id_mesure               int64
date                   object
batiment               object
type_batiment          object
zone                   object
temperature           float64
humidite              float64
co2                   float64
occupation            float64
consommation_kwh      float64
mode_climatisation     object
etat_systeme           object
jour_semaine           object
alerte                 object
dtype: object

In [48]:
# Question 9 : Identifier les identifiants
print("Nombre de lignes :", df.shape[0])
print("Valeurs uniques id_mesure :", df['id_mesure'].nunique())
print("Valeurs uniques batiment :", df['batiment'].nunique())

# id_mesure est l'identifiant du dataset, bien qu'il contienne quelques doublons
# (507 lignes vs 500 valeurs uniques) 

Nombre de lignes : 507
Valeurs uniques id_mesure : 500
Valeurs uniques batiment : 8


In [49]:
# Question 10 : Statistiques descriptives (moyenne, médiane, min, max, écart-type, quartiles)
df.describe()

,id_mesure,temperature,humidite,co2,occupation,consommation_kwh
count,507.000000,495.000000,496.000000,500.000000,501.000000,502.000000
mean,1251.114398,24.154141,57.864113,844.150000,44.850299,169.069323
std,144.782769,7.418465,16.026336,582.181386,24.949139,53.164294
min,1001.000000,-30.000000,-12.000000,89.000000,-20.000000,-100.000000
25%,1125.500000,21.600000,49.275000,623.750000,27.000000,136.875000
50%,1252.000000,24.000000,57.550000,787.500000,46.000000,169.800000
75%,1376.500000,26.000000,65.750000,952.000000,61.000000,202.975000
max,1500.000000,96.000000,160.000000,6000.000000,116.000000,336.200000


In [50]:
# Question 11 : Variables potentiellement problématiques

**Variables potentiellement problématiques identifiées :**

- **temperature** : valeurs manquantes (12) + valeurs aberrantes (min = -30°C, max = 96°C, physiquement impossibles pour un bâtiment)
- **humidite** : valeurs manquantes (11) + valeurs incohérentes (min = -12, max = 160 ; une humidité relative doit être comprise entre 0 et 100%)
- **co2** : valeurs manquantes (7) + valeur aberrante extrême (max = 6000, très au-dessus de la moyenne de ~844 et du 75e percentile de 952)
- **occupation** : valeurs manquantes (6) + valeur négative impossible (min = -20 ; un nombre de personnes ne peut pas être négatif)
- **consommation_kwh** : valeurs manquantes (5) + valeur négative impossible (min = -100)
- **id_mesure** : contient des doublons (500 valeurs uniques pour 507 lignes) alors qu'il devrait être unique

In [51]:
# Question 12.a : Rechercher des valeurs telles que humidité < 0
df[df['humidite'] < 0]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
281,1126,2025-02-01 06:00:00,B2,École,D,24.8,-5.0,759.0,57.0,225.2,Boost,Normal,Samedi,Non
335,1036,2025-01-09 18:00:00,B4,Bureau,D,21.2,-8.0,160.0,25.0,120.2,Normal,Alerte,Jeudi,Non
366,1216,2025-02-23 18:00:00,B3,Hôpital,B,23.2,-12.0,702.0,41.0,119.8,Normal,Normal,Dimanche,Non


In [52]:
# Question 12.b : Rechercher des valeurs telles que humidité > 100
df[df['humidite'] > 100]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
59,1246,2025-03-03 06:00:00,B5,Centre commercial,A,24.2,108.0,813.0,18.0,-15.0,Normal,Normal,Lundi,Non
103,1016,2025-01-04 18:00:00,B6,Université,B,26.9,145.0,670.0,28.0,175.9,Normal,Normal,Samedi,Non
128,1156,2025-02-08 18:00:00,B1,Bureau,A,NaN,160.0,941.0,37.0,149.8,Eco,Normal,Samedi,Oui
160,1186,2025-02-16 06:00:00,B8,Entrepôt,B,24.9,125.0,NaN,18.0,123.5,Normal,Normal,Dimanche,Non
268,1276,2025-03-10 18:00:00,B1,Bureau,D,26.0,140.0,633.0,25.0,148.3,Normal,Normal,Lundi,Non
327,1066,2025-01-17 06:00:00,B2,École,C,25.8,132.0,646.0,14.0,93.6,Eco,Normal,Vendredi,Non
342,1096,2025-01-24 18:00:00,B8,Entrepôt,A,20.3,110.0,783.0,55.0,121.0,Normal,Normal,Vendredi,Non


In [53]:
# Question 12.c : Rechercher des valeurs de température extrêmement élevée
df[df['temperature'] > 50]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
7,1141,2025-02-05 00:00:00,B2,École,D,72.5,78.3,257.0,100.0,239.7,Boost,Alerte,Mercredi,Non
116,1181,2025-02-15 00:00:00,B6,Université,A,96.0,86.5,874.0,115.0,241.5,Normal,Normal,Samedi,Non
161,1061,2025-01-16 00:00:00,B5,Centre commercial,D,88.0,50.5,393.0,62.0,196.6,Normal,Alerte,Jeudi,Non
499,1021,2025-01-06 00:00:00,B2,École,A,95.2,65.0,314.0,43.0,246.3,Normal,Normal,Lundi,Non


In [ ]:
# Question 12.d : Rechercher des valeurs de CO2 négatives
df[df['co2'] < 0]
# Aucune valeur négative trouvée pour le CO2

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
